In [14]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
data = r"G:\My Drive\New_Working_Files\KNIME\CTU_2026\ML_Models\Order_raw_data.csv"
df = pd.read_csv(data).replace(0, np.nan)
print(df.columns)

Index(['oms_version', 'id', 'order_item_id', 'type', 'flow_type',
       'order_external_id', 'checkout_id', 'status', 'net_filter_flag', 'gmv',
       ...
       'updated_at', 'brand_classification_key', 'is_large', 'cms_vertical',
       'is_bulk_order', 'order_sales_app', 'order_sales_experience',
       'order_tags', 'unit_tags', 'is_extra_saver_flag'],
      dtype='object', length=227)


In [15]:
df['unit_creation_date_time'] = pd.to_datetime(df['unit_creation_date_time'])
df['order_week'] = df['unit_creation_date_time'].dt.isocalendar().week
dim_product = df[['product_id', 'product_title', 'brand', 'analytic_category', 'mrp']].drop_duplicates(subset=['product_id'])
dim_seller = df[['seller_id', 'seller_id_key', 'is_alpha_seller', 'cluster']].drop_duplicates(subset=['seller_id'])
fact_orders = df[['order_item_id', 'product_id', 'seller_id', 'gmv', 'discount_value', 'order_week', 'status']]

analysis_df = fact_orders.merge(dim_product, on='product_id', how="left").merge(dim_seller, on ='seller_id', how ="left")

In [53]:
weekly_brand_trend = (analysis_df.groupby(['order_week', 'brand'])['gmv']
                      .sum()
                      .reset_index()
                     )


weekly_top5_per_week  = (
    weekly_brand_trend.groupby('order_week', group_keys=False)
    .apply(lambda g: g.nlargest(5, 'gmv'))
    .reset_index(drop=True))

print(weekly_top5_per_week.sort_values(['order_week','gmv'], ascending=[True, False]))

output= r"G:\My Drive\New_Working_Files\KNIME\CTU_2026\ML_Models\Output_gmv.xlsx"
weekly_top5_per_week.to_excel(output, index=False)
#top_5brands = (weekly_brand_trend.groupby('brand')['gmv']
 #              .sum()
  #             .sort_values(ascending=False)
   #            .head(5)
    #           .index
     #         )

#weekly_top5 = weekly_brand_trend[weekly_brand_trend['brand'].isin(top_5brands)]
#print(weekly_top5.sort_values(['order_week', 'gmv'], ascending=[True, False]))

#brand_totals  = weekly_brand_trend.sum(axis=0)
#top5_brands = brand_totals.sort_values(ascending=False).head(5).index
#weekly_brand_trend.columns = weekly_brand_trend.columns.get_level_values(1)
#brand_totals  = weekly_brand_trend.sum(axis=0)



     order_week        brand      gmv
0             1     Prestige   4165.0
1             1      CHKOKKO   2216.0
2             1       PetJoy   1227.0
3             1        HARPA   1141.0
4             1         boAt   1105.0
..          ...          ...      ...
123          51  Allen Solly   1196.0
124          51    Matraders     28.0
125          52    Aquaguard  20105.0
126          52       I Kall   1606.0
127          52        RINTL    217.0

[128 rows x 3 columns]


In [48]:
print(weekly_top5)

        order_week     brand        gmv
713              3   Samsung     1265.0
930              3    realme    86187.0
949              3      vivo    32115.0
1297             4     Apple   589031.0
3591             4  MOTOROLA   262363.0
...            ...       ...        ...
103624          17  MOTOROLA   942275.0
105053          17   Samsung   341961.0
106543          17    realme   597877.0
106667          17      vivo  1004152.0
106699          23   Samsung        0.0

[74 rows x 3 columns]


In [58]:
weekly_seller_quality = analysis_df.groupby(['order_week', 'is_alpha_seller']).agg({
    'status': lambda x: (x == 'DELIVERED').sum() / len(x) *100 
}).rename(columns={'status': 'cancellation_rate_percent'})

#print("Weekly brand gmv trend")
#print(weekly_brand_trend.fillna(0).round(2))

print("\n Weekly seller quality")
print(weekly_seller_quality)


 Weekly seller quality
                            cancellation_rate_percent
order_week is_alpha_seller                           
1          True                              0.000000
2          True                              0.000000
3          True                             54.954955
4          True                             90.134401
5          True                             91.668010
6          True                             90.810544
7          True                             91.071121
8          True                             91.262755
9          True                             91.654098
10         True                             91.252840
11         True                             91.525006
12         True                             92.580238
13         True                             92.126323
14         True                             92.144077
15         True                             92.684961
16         True                             92.129866
17  

In [59]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [62]:
data = r"G:\My Drive\New_Working_Files\KNIME\CTU_2026\ML_Models\Order_raw_data.csv"
df7 = pd.read_csv(data).fillna(0)


In [66]:
features = [
    'gmv', 'listing_tier', 'listing_type', 'sla_in_days',
    'order_status', 'shipping_charge_adjustment', 'discount_value'
]

X = df[features]
y= (df['status'] =="DELIVERED").astype(int)

#X=pd.get_dummies(X, drop_first=True)
preprocessor = 

X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators = 100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.44      0.32      0.37     14223
           1       0.81      0.87      0.84     45778

    accuracy                           0.74     60001
   macro avg       0.62      0.60      0.61     60001
weighted avg       0.72      0.74      0.73     60001

